# Active Experimental Design — Step-by-Step Smart Agent
## CAMM Hackathon Day 13 — Problem 2

**Workflow:**
1. Start with **3 seed measurements** at random locations
2. Fit a Gaussian Process to the data
3. **Smart agent** inspects the GP and decides: *where* to measure next and *how long*
4. Take the measurement, update the GP
5. **Plot the full state** — showing the agent decision before and after
6. Repeat until budget is exhausted

**Every single iteration produces a plot.** You can watch the GP improve measurement by measurement.

In [ ]:
!pip install scikit-learn -q
print('Ready')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
import warnings, copy
warnings.filterwarnings('ignore')

P = {
    'blue':   '#2563EB',
    'green':  '#16A34A',
    'red':    '#DC2626',
    'amber':  '#D97706',
    'purple': '#7C3AED',
    'slate':  '#64748B',
    'bg':     '#F8FAFC',
    'grid':   '#CBD5E1',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   P['bg'],
    'axes.grid':        True,
    'grid.color':       P['grid'],
    'grid.linewidth':   0.8,
    'font.size':        11,
    'axes.labelsize':   12,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'legend.framealpha':  0.92,
    'legend.fontsize':    9,
    'figure.dpi':         110,
})
print('Imports and style ready')

## Target Functions, Noise, Simulator

In [ ]:
def multimodal(x):
    x = np.asarray(x)
    return (10 + x**2 - 7*np.cos(2*np.pi*x) + 9*np.sin(1.4*np.pi*x)) / 40

def unimodal(x):
    return 1.2 * np.exp(-0.5*((x-2.3)/0.7)**2) + 0.1

def oscillatory(x):
    return 0.55 + 0.15*np.sin(1.8*x) + 0.08*np.cos(6.0*x)

def double_well(x):
    x = np.asarray(x)
    return (0.08*(x**2 - 7)**2 + 0.45*x) / 10

def broad_narrow(x):
    x = np.asarray(x)
    return (1 - 1.0*np.exp(-((x-2.5)**2)/(2*0.9**2))
              - 1.6*np.exp(-((x-1.0)**2)/(2*0.08**2)))

def noise_decay_floor(x=None, t=None, sigma_min=0.05, A=0.18, tau=1.0):
    return sigma_min + A * np.exp(-np.asarray(t) / tau)


class MeasurementSimulator:
    def __init__(self, f, noise_fn, rng=42):
        self.f = f
        self.noise_fn = noise_fn
        self.rng = np.random.default_rng(rng)

    def measure(self, x, t):
        sigma = float(self.noise_fn(x=x, t=np.array([t])))
        return float(self.f(np.array([x]))) + self.rng.normal(0.0, sigma)

print('Functions ready')

## GP Surrogate & Budget Tracker

In [ ]:
class GPSurrogate:
    def __init__(self):
        kernel = C(1.0,(0.01,10)) * RBF(0.5,(0.05,2.0)) + WhiteKernel(0.01,(1e-5,0.3))
        self.gp = GaussianProcessRegressor(
            kernel=kernel, n_restarts_optimizer=5,
            normalize_y=True, alpha=1e-6
        )
        self.x_obs, self.y_obs = [], []
        self.fitted = False

    def add(self, x, y):
        self.x_obs.append(float(x))
        self.y_obs.append(float(y))
        if len(self.x_obs) >= 2:
            self.gp.fit(np.array(self.x_obs).reshape(-1,1), np.array(self.y_obs))
            self.fitted = True

    def predict(self, xg):
        if not self.fitted:
            return np.zeros_like(xg), np.ones_like(xg)
        return self.gp.predict(xg.reshape(-1,1), return_std=True)

    def mse(self, xg, f):
        m, _ = self.predict(xg)
        return float(np.mean((m - f(xg))**2))

    def mean_std(self, xg):
        _, s = self.predict(xg)
        return float(np.mean(s))


class BudgetTracker:
    def __init__(self, total=15.0, tw=0.15):
        self.total, self.tw = total, tw
        self.spent = 0.0
        self.x_curr = None
        self.log = []

    def travel(self, x):
        return self.tw * abs(x - self.x_curr) if self.x_curr is not None else 0.0

    def can(self, x, dur):
        return self.spent + dur + self.travel(x) <= self.total

    def spend(self, x, dur, it, reason=''):
        tc = self.travel(x)
        sc = dur + tc
        self.spent += sc
        self.log.append(dict(
            iteration=it, x=float(x), duration=float(dur),
            travel=tc, acq=float(dur), step=sc,
            cumulative=self.spent, remaining=self.total - self.spent,
            reason=reason,
        ))
        self.x_curr = x
        return sc

    @property
    def remaining(self): return self.total - self.spent

print('GP and Budget Tracker ready')

## Smart Agent Decision Function

At each step the agent scores every candidate x by:

```
score(x) = uncertainty(x)  -  0.35 * travel_cost(x)  -  0.25 * proximity_to_existing(x)
```

- **Uncertainty** drives exploration toward unknown regions
- **Travel cost** penalises far-away positions (saves budget)
- **Proximity penalty** prevents redundant measurements in already-sampled areas

Acquisition time is adapted to the remaining budget.

In [ ]:
def agent_decide(gp, bud, xg, min_spacing=0.15):
    """
    Smart agent decision.
    Returns (x_next, duration, reasoning_string)
    """
    _, std = gp.predict(xg)

    # Proximity penalty: repel from already-measured locations
    prox = np.zeros_like(xg)
    for xo in gp.x_obs:
        prox += np.exp(-np.abs(xg - xo) / 0.4)

    # Travel penalty
    travel_pen = np.array([bud.travel(float(x)) for x in xg])

    # Combined score
    scores = std - 0.35 * travel_pen - 0.25 * prox

    # Mask positions too close to existing observations
    for xo in gp.x_obs:
        scores[np.abs(xg - xo) < min_spacing] = -np.inf

    best_idx = int(np.argmax(scores))
    x_next   = float(xg[best_idx])
    unc_next = float(std[best_idx])
    prox_val = float(prox[best_idx])
    trav_val = bud.travel(x_next)

    # Adaptive duration based on remaining budget
    if   bud.remaining > 10: dur, strategy = 1.2, 'ample budget — long acq for high accuracy'
    elif bud.remaining > 7:  dur, strategy = 0.9, 'good budget — balanced measurement'
    elif bud.remaining > 4:  dur, strategy = 0.6, 'moderate budget — efficient sampling'
    elif bud.remaining > 2:  dur, strategy = 0.35,'low budget — quick probe'
    else:                     dur, strategy = 0.15,'critical budget — minimal acquisition'

    dur = float(np.clip(dur, 0.1, bud.remaining * 0.5))

    reasoning = (
        f'Chose x={x_next:.3f} '
        f'[unc={unc_next:.4f}, travel={trav_val:.3f}, prox={prox_val:.3f}]. '
        f'Duration={dur:.2f}s. Strategy: {strategy}. '
        f'Budget left={bud.remaining:.2f}/{bud.total:.1f}'
    )

    return x_next, dur, reasoning

print('Smart agent ready')

## Per-Step Plot Function

Each call produces a **4-panel figure**:
- Left (large): GP posterior — true function, GP mean, uncertainty bands, all measurements, and the **next planned measurement** marked with a star
- Top right: MSE over iterations
- Middle right: Uncertainty over iterations  
- Bottom right: Budget gauge + agent reasoning

In [ ]:
def plot_step(gp, bud, xg, f_true, iteration,
              x_next=None, dur_next=None, reason=None,
              mse_history=None, unc_history=None,
              phase='decision'):
    """
    Professional per-iteration figure.
    phase='decision' : shows the agent's planned next point (green star)
    phase='result'   : shows the result after measurement (no green star)
    """
    fig = plt.figure(figsize=(18, 7))
    gs  = gridspec.GridSpec(
        3, 2, figure=fig,
        width_ratios=[3, 1.4],
        height_ratios=[1, 1, 1],
        hspace=0.55, wspace=0.35
    )

    ax_gp  = fig.add_subplot(gs[:, 0])    # full left column
    ax_mse = fig.add_subplot(gs[0, 1])
    ax_unc = fig.add_subplot(gs[1, 1])
    ax_inf = fig.add_subplot(gs[2, 1])

    # ── GP posterior ──────────────────────────────────────────────────
    m, s   = gp.predict(xg)
    y_true = f_true(xg)

    ax_gp.plot(xg, y_true, '--', color=P['slate'], lw=1.8,
               alpha=0.65, label='Ground truth', zorder=1)
    ax_gp.plot(xg, m, color=P['blue'], lw=2.8,
               label='GP mean', zorder=3)
    ax_gp.fill_between(xg, m-2*s, m+2*s, alpha=0.13,
                       color=P['blue'], label='95% CI', zorder=2)
    ax_gp.fill_between(xg, m-s, m+s, alpha=0.22,
                       color=P['blue'], label='68% CI', zorder=2)

    # Numbered observation points
    if gp.x_obs:
        ax_gp.scatter(gp.x_obs, gp.y_obs, color=P['red'], s=110,
                      zorder=5, edgecolors='white', lw=2,
                      label=f'Measurements (n={len(gp.x_obs)})')
        for i, (xo, yo) in enumerate(zip(gp.x_obs, gp.y_obs)):
            ax_gp.annotate(
                str(i+1), (xo, yo),
                textcoords='offset points', xytext=(0, 11),
                ha='center', fontsize=8.5,
                color=P['red'], fontweight='bold'
            )

    # Next measurement marker (decision phase only)
    if x_next is not None and phase == 'decision':
        gp_at_next = float(m[np.argmin(np.abs(xg - x_next))])
        ax_gp.axvline(x_next, color=P['green'], ls=':', lw=2.5, alpha=0.75, zorder=4)
        ax_gp.scatter([x_next], [gp_at_next], color=P['green'], s=320,
                      zorder=6, marker='*', edgecolors='white', lw=1.5,
                      label=f'Next: x={x_next:.3f}, t={dur_next:.2f}s')
        offset_y = 0.18 * (m.max() - m.min()) + 0.05
        offset_x = 0.3 if x_next < 4.0 else -0.6
        ax_gp.annotate(
            f'NEXT TEST\nx = {x_next:.3f}\nt = {dur_next:.2f}s',
            xy=(x_next, gp_at_next),
            xytext=(x_next + offset_x, gp_at_next + offset_y),
            arrowprops=dict(arrowstyle='->', color=P['green'], lw=2),
            fontsize=9.5, color=P['green'], fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', fc='white',
                      ec=P['green'], alpha=0.95)
        )

    mse = gp.mse(xg, f_true)
    phase_label = 'Agent Decision' if phase == 'decision' else 'After Measurement'
    ax_gp.set_title(
        f'Iteration {iteration} [{phase_label}]  |  '
        f'n={len(gp.x_obs)} measurements  |  '
        f'MSE = {mse:.5f}  |  '
        f'Budget = {bud.remaining:.2f} / {bud.total:.1f}',
        fontsize=12, pad=10
    )
    ax_gp.set_xlabel('x'); ax_gp.set_ylabel('f(x)')
    ax_gp.set_xlim(-0.1, 5.1)
    ax_gp.legend(loc='upper right', fontsize=8.5)

    # ── Running MSE ───────────────────────────────────────────────────
    if mse_history:
        iters = list(range(1, len(mse_history)+1))
        ax_mse.plot(iters, mse_history, color=P['red'], lw=2.5, marker='o', ms=6)
        ax_mse.fill_between(iters, 0, mse_history, alpha=0.15, color=P['red'])
        ax_mse.scatter([iters[-1]], [mse_history[-1]], color=P['red'], s=100, zorder=5)
    ax_mse.set_xlabel('Iteration'); ax_mse.set_ylabel('MSE')
    ax_mse.set_title('MSE Progress')

    # ── Running uncertainty ───────────────────────────────────────────
    if unc_history:
        iters = list(range(1, len(unc_history)+1))
        ax_unc.plot(iters, unc_history, color=P['purple'], lw=2.5, marker='s', ms=6)
        ax_unc.fill_between(iters, 0, unc_history, alpha=0.15, color=P['purple'])
        ax_unc.scatter([iters[-1]], [unc_history[-1]], color=P['purple'], s=100, zorder=5)
    ax_unc.set_xlabel('Iteration'); ax_unc.set_ylabel('Mean Std')
    ax_unc.set_title('Uncertainty Progress')

    # ── Budget + reasoning ────────────────────────────────────────────
    ax_inf.set_xlim(0, 1); ax_inf.set_ylim(0, 1); ax_inf.axis('off')
    pct_spent = min(bud.spent / bud.total, 1.0)
    ax_inf.barh([0.88], [1.0],   height=0.09, color=P['grid'],  alpha=0.5)
    ax_inf.barh([0.88], [pct_spent], height=0.09, color=P['amber'], alpha=0.85)
    ax_inf.text(0.5, 0.96,
               f'Budget: {bud.remaining:.2f}/{bud.total:.1f} ({100*(1-pct_spent):.0f}% left)',
               ha='center', va='top', fontsize=9, fontweight='bold')

    if reason:
        words  = reason.split()
        lines, cur = [], []
        for w in words:
            cur.append(w)
            if len(' '.join(cur)) > 36:
                lines.append(' '.join(cur[:-1]))
                cur = [w]
        lines.append(' '.join(cur))
        ax_inf.text(0.04, 0.74, 'Agent reasoning:',
                   fontsize=9, fontweight='bold', color=P['blue'], va='top')
        ax_inf.text(0.04, 0.64, '\n'.join(lines[:9]),
                   fontsize=7.8, va='top', color=P['slate'], family='monospace')

    plt.suptitle('Step-by-Step Smart Agent — Active Experimental Design',
                fontsize=13, fontweight='bold', y=1.01)

    tag = 'a_decision' if phase == 'decision' else 'b_result'
    plt.savefig(f'step_{iteration:02d}_{tag}.png', bbox_inches='tight', dpi=125)
    plt.show()
    plt.close()

print('plot_step ready')

## Configuration — Edit only this cell

In [ ]:
# -------------------------------------------------------
F_TRUE       = multimodal      # unimodal | oscillatory | double_well | broad_narrow
TOTAL_BUDGET = 15.0
N_INIT       = 3               # number of seed measurements
SEED         = 42
# -------------------------------------------------------

X_GRID = np.linspace(0, 5, 300)
sim    = MeasurementSimulator(f=F_TRUE, noise_fn=noise_decay_floor, rng=SEED)

print(f'Target : {F_TRUE.__name__}')
print(f'Budget : {TOTAL_BUDGET}')
print(f'Seeds  : {N_INIT}')

## Phase 1 — Seed Measurements (n=3)

In [ ]:
gp  = GPSurrogate()
bud = BudgetTracker(total=TOTAL_BUDGET)

mse_history = []
unc_history = []
step_log    = []

print(f'=== Seeding: {N_INIT} initial measurements ===')
rng_init    = np.random.default_rng(SEED)
init_xs     = rng_init.uniform(0.5, 4.5, N_INIT)

for i, x0 in enumerate(init_xs):
    dur = 0.6
    if bud.can(x0, dur):
        y = sim.measure(x0, dur)
        gp.add(x0, y)
        bud.spend(x0, dur, i, reason=f'seed measurement {i+1}')
        print(f'  Seed {i+1}: x={x0:.3f}  y={y:.4f}  dur={dur}s')

mse_history.append(gp.mse(X_GRID, F_TRUE))
unc_history.append(gp.mean_std(X_GRID))

print(f'\nSeeding done. n={len(gp.x_obs)}, MSE={mse_history[-1]:.5f}, '
      f'budget_left={bud.remaining:.2f}')

# Plot initial state
plot_step(
    gp, bud, X_GRID, F_TRUE,
    iteration=0,
    x_next=None, reason='Seed phase complete. Agent begins decision loop next.',
    mse_history=mse_history, unc_history=unc_history,
    phase='result'
)

## Phase 2 — Agent Loop

For each iteration:
1. Agent inspects GP and decides next test → **plot shows the decision (green star)**
2. Measurement is taken and GP is updated → **plot shows the result**

In [ ]:
print('=== Smart Agent Active Learning Loop ===\n')

iteration = 1

while bud.remaining > 0.5:

    # ── 1. Agent decides ──────────────────────────────────────
    x_next, dur_next, reason = agent_decide(gp, bud, X_GRID)

    if not bud.can(x_next, dur_next):
        print(f'Iteration {iteration}: budget too low to continue.')
        break

    print(f'Iteration {iteration}')
    print(f'  Agent chose  x={x_next:.3f}  dur={dur_next:.2f}s')
    print(f'  Reason: {reason[:90]}')

    # ── 2. Plot BEFORE measurement (agent decision) ────────────
    plot_step(
        gp, bud, X_GRID, F_TRUE,
        iteration=iteration,
        x_next=x_next, dur_next=dur_next, reason=reason,
        mse_history=list(mse_history),
        unc_history=list(unc_history),
        phase='decision'
    )

    # ── 3. Take the measurement ───────────────────────────────
    y = sim.measure(x_next, dur_next)
    gp.add(x_next, y)
    bud.spend(x_next, dur_next, iteration, reason)

    new_mse = gp.mse(X_GRID, F_TRUE)
    new_unc = gp.mean_std(X_GRID)
    mse_history.append(new_mse)
    unc_history.append(new_unc)

    step_log.append({
        'iteration': iteration,
        'x': x_next, 'duration': dur_next, 'y': y,
        'reasoning': reason,
        'mse': new_mse,
        'mean_uncertainty': new_unc,
        'budget_remaining': bud.remaining,
    })

    # ── 4. Plot AFTER measurement (result) ───────────────────
    plot_step(
        gp, bud, X_GRID, F_TRUE,
        iteration=iteration,
        x_next=None, reason=reason,
        mse_history=list(mse_history),
        unc_history=list(unc_history),
        phase='result'
    )

    print(f'  Measured y={y:.4f}  new MSE={new_mse:.5f}  budget_left={bud.remaining:.2f}\n')

    iteration += 1

print('=== Loop complete ===')
print(f'Total measurements : {len(gp.x_obs)}')
print(f'Final MSE          : {gp.mse(X_GRID, F_TRUE):.5f}')
print(f'Budget remaining   : {bud.remaining:.2f}')

## Final Summary Plots

In [ ]:
# Final GP fit
fig, ax = plt.subplots(figsize=(13, 5))
m, s = gp.predict(X_GRID)

ax.plot(X_GRID, F_TRUE(X_GRID), '--', color=P['slate'], lw=2,
        alpha=0.7, label='Ground truth')
ax.plot(X_GRID, m, color=P['blue'], lw=3, label='Final GP mean')
ax.fill_between(X_GRID, m-2*s, m+2*s, alpha=0.15, color=P['blue'], label='95% CI')
ax.fill_between(X_GRID, m-s,   m+s,   alpha=0.25, color=P['blue'], label='68% CI')
ax.scatter(gp.x_obs, gp.y_obs, color=P['red'], s=120, zorder=5,
           edgecolors='white', lw=2, label=f'All measurements (n={len(gp.x_obs)})')

for i, (xo, yo) in enumerate(zip(gp.x_obs, gp.y_obs)):
    ax.annotate(str(i+1), (xo, yo),
               textcoords='offset points', xytext=(0, 11),
               ha='center', fontsize=9, color=P['red'], fontweight='bold')

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title(
    f'Final GP Model  |  n={len(gp.x_obs)} measurements  |  '
    f'MSE={gp.mse(X_GRID,F_TRUE):.5f}  |  '
    f'Budget used={bud.spent:.2f}/{bud.total:.1f}',
    fontsize=13
)
ax.legend(); ax.set_xlim(0, 5)
plt.tight_layout()
plt.savefig('final_gp_fit.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Convergence + measurement sequence
if step_log:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    its  = [s['iteration']        for s in step_log]
    mses = [s['mse']              for s in step_log]
    uncs = [s['mean_uncertainty'] for s in step_log]
    durs = [s['duration']         for s in step_log]
    xpos = [s['x']                for s in step_log]

    axes[0].plot(its, mses, color=P['red'], lw=2.5, marker='o', ms=7)
    axes[0].fill_between(its, 0, mses, alpha=0.15, color=P['red'])
    axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('MSE vs Ground Truth')
    axes[0].set_title('MSE Convergence')

    axes[1].plot(its, uncs, color=P['purple'], lw=2.5, marker='s', ms=7)
    axes[1].fill_between(its, 0, uncs, alpha=0.15, color=P['purple'])
    axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Mean Posterior Std')
    axes[1].set_title('Uncertainty Reduction')

    sc = axes[2].scatter(its, xpos, c=durs, cmap='YlOrRd',
                         s=120, edgecolors='white', lw=1.5, zorder=5)
    axes[2].plot(its, xpos, color=P['slate'], lw=1, alpha=0.4)
    plt.colorbar(sc, ax=axes[2], label='Duration (s)', shrink=0.85)
    axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('x position')
    axes[2].set_title('Agent Measurement Sequence')
    axes[2].set_ylim(-0.2, 5.2)

    fig.suptitle('Active Learning Summary — Smart Agent',
                fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('final_summary.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Budget breakdown
if bud.log:
    fig = plt.figure(figsize=(14, 4.5))
    gs  = gridspec.GridSpec(1, 3, wspace=0.38)
    ax1, ax2, ax3 = (fig.add_subplot(gs[i]) for i in range(3))

    log  = bud.log
    its  = [r['iteration']  for r in log]
    cum  = [r['cumulative'] for r in log]
    acq  = [r['acq']        for r in log]
    trav = [r['travel']     for r in log]

    ax1.plot(its, cum, color=P['red'], lw=2.5, marker='o', ms=6)
    ax1.axhline(TOTAL_BUDGET, color=P['slate'], ls='--', lw=1.5, label='Limit')
    ax1.fill_between(its, 0, cum, alpha=0.15, color=P['red'])
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Cumulative Cost')
    ax1.set_title('Budget Consumption'); ax1.legend()

    x = np.arange(len(its))
    ax2.bar(x, acq,  label='Acquisition', color=P['blue'],  alpha=0.85)
    ax2.bar(x, trav, bottom=acq, label='Travel', color=P['amber'], alpha=0.85)
    step = max(1, len(its)//8)
    ax2.set_xticks(x[::step]); ax2.set_xticklabels(its[::step])
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Step Cost')
    ax2.set_title('Cost Breakdown'); ax2.legend()

    ta = sum(acq); tt = sum(trav); un = max(0, TOTAL_BUDGET - bud.spent)
    ax3.pie(
        [ta, tt, un],
        labels=[f'Acquisition\n{ta:.2f}', f'Travel\n{tt:.2f}', f'Unused\n{un:.2f}'],
        colors=[P['blue'], P['amber'], P['grid']],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(width=0.6), pctdistance=0.75
    )
    ax3.set_title('Budget Allocation')

    fig.suptitle('Budget Analysis', fontsize=14, fontweight='bold')
    plt.savefig('budget_analysis.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Agent reasoning log
print('=== Agent Reasoning Log ===\n')
print(f'{"Step":>4}  {"x":>7}  {"dur":>5}  {"y":>7}  {"MSE":>9}  {"Bud left":>8}')
print('-' * 70)
for s in step_log:
    print(f"{s['iteration']:>4}  {s['x']:>7.3f}  {s['duration']:>5.2f}  "
          f"{s['y']:>7.4f}  {s['mse']:>9.5f}  {s['budget_remaining']:>8.2f}")
    print(f"       -> {s['reasoning'][:95]}")
    print()